In [1]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
import time

In [2]:
zs = pd.read_table('/oak/stanford/groups/jpriest/cnv_ukb/cnv_constraint_zscores_20190430.tsv',
                   names=['Gene','z','pLI'], 
                   index_col='Gene')
zs.head()

,z,pLI
Gene,,
BRCA2,3.402038,0.991096
BRCA1,2.570469,0.984025
APC,2.085995,0.945645
ATM,1.242243,0.989235
MSH2,1.240663,0.988295


In [3]:
sets = pd.read_table('/oak/stanford/groups/jpriest/cnv_ukb/resources/HPO_pheno_to_gene.txt', skiprows=0,
                     names=['hpoID','hpolabel','geneID','Gene'])
sets.head()

,hpoID,hpolabel,geneID,Gene
0,HP:0001459,1-3 toe syndactyly,2737,GLI3
1,HP:0006088,1-5 finger complete cutaneous syndactyly,64327,LMBR1
2,HP:0010708,1-5 finger syndactyly,6469,SHH
3,HP:0010708,1-5 finger syndactyly,64327,LMBR1
4,HP:0010713,1-5 toe syndactyly,2737,GLI3


In [4]:
sets['hpoID'].value_counts().shape

(8524,)

In [5]:
load = True
if load:
    hpo_t = pd.read_table("/oak/stanford/groups/jpriest/cnv_ukb/cnv_constraint_hpo_enrichment_20190430.tsv")
else:
    hpo_t = pd.DataFrame()
    t = time.time()
    for hpo in sets['hpoID'].value_counts().index:
        # select rows
        in_set = sets.query('hpoID == @hpo and Gene in @zs.index')
        out_of_set = sets.query('hpoID != @hpo and Gene in @zs.index')
        if in_set.shape[0] == 0 or out_of_set.shape[0] == 0:
            continue
        # select genes
        in_set_z = zs.loc[in_set['Gene'],'z']
        out_of_set_z = zs.loc[out_of_set['Gene'],'z']
        # compute mean shift, t-test p-value
        deltaZ = in_set_z.mean() - out_of_set_z.mean()
        _,p = ttest_ind(in_set_z.values, out_of_set_z.values, equal_var=False, nan_policy='omit')
        # append
        hpo_t = hpo_t.append([[hpo, in_set.iloc[0,:]['hpolabel'], deltaZ, p, in_set_z.shape[0]]])
        if hpo_t.shape[0] % 200 == 0:
            print(hpo_t.shape[0], time.time()-t)
            t=time.time()
    hpo_t.columns = ['hpoID','hpoLabel','deltaZ','pValue','nGenes']
hpo_t.head()

,hpoID,hpoLabel,deltaZ,pValue,nGenes
0,HP:0000707,Abnormality of the nervous system,-0.007672,0.009988,2619
1,HP:0012638,Abnormality of nervous system physiology,-0.007689,0.014391,2433
2,HP:0000152,Abnormality of head or neck,-0.004238,0.233858,2115
3,HP:0000234,Abnormality of the head,-0.004255,0.235575,2094
4,HP:0000924,Abnormality of the skeletal system,-0.006123,0.088247,2047


In [6]:
hpo_t['pValue'] = hpo_t['pValue'].astype(float)

In [7]:
hpo_t.query('nGenes >= 5 and deltaZ > 0').sort_values('pValue').head(20)

,hpoID,hpoLabel,deltaZ,pValue,nGenes
1178,HP:0002034,Abnormality of the rectum,0.091344,0.005468,74
910,HP:0100006,Neoplasm of the central nervous system,0.084712,0.005929,107
452,HP:0011792,Neoplasm by histology,0.065458,0.006319,228
549,HP:0002027,Abdominal pain,0.084144,0.006739,177
1321,HP:0100834,Neoplasm of the large intestine,0.206365,0.006946,67
1612,HP:0100273,Neoplasm of the colon,0.277430,0.007167,49
904,HP:0010787,Genital neoplasm,0.133163,0.008309,102
730,HP:0008069,Neoplasm of the skin,0.064637,0.008358,140
973,HP:0100242,Sarcoma,0.084586,0.008506,103
573,HP:0000953,Hyperpigmentation of the skin,0.079611,0.008726,178


In [15]:
hpo_t.query('nGenes >= 5 and deltaZ < 0').sort_values('pValue').head(50)

,hpoID,hpoLabel,deltaZ,pValue,nGenes
225,HP:0000648,Optic atrophy,-0.028272,5.787128e-13,430
651,HP:0012072,Aciduria,-0.036503,3.258807e-12,155
614,HP:0001344,Absent speech,-0.034383,2.544988e-10,158
208,HP:0012795,Abnormality of the optic disc,-0.026035,3.251769e-10,467
1019,HP:0009136,Duplication involving bones of the feet,-0.044283,4.473983e-09,92
832,HP:0003355,Aminoaciduria,-0.035638,7.014999e-09,115
1048,HP:0001829,Foot polydactyly,-0.043610,1.315878e-08,88
545,HP:0010442,Polydactyly,-0.032821,4.196554e-08,182
330,HP:0005280,Depressed nasal bridge,-0.026340,4.609781e-08,303
1119,HP:0001162,Postaxial hand polydactyly,-0.035859,6.691444e-08,80


In [9]:
if not load:
    hpo_t.to_csv('/oak/stanford/groups/jpriest/cnv_ukb/cnv_constraint_hpo_enrichment_20181217.tsv', sep='\t')

In [10]:
hpo_t.query('deltaZ > 0 and pValue < 0.01').sort_values('deltaZ', ascending=False).head(20)

,hpoID,hpoLabel,deltaZ,pValue,nGenes
6280,HP:0006778,Benign genitourinary tract neoplasm,1.165369,0.004459,2
6141,HP:0006719,Benign gastrointestinal tract tumors,1.165369,0.004459,2
6137,HP:0030410,Sebaceous gland carcinoma,1.165369,0.004459,2
5318,HP:0006758,Malignant genitourinary tract tumor,1.073453,0.007271,3
1612,HP:0100273,Neoplasm of the colon,0.277430,0.007167,49
1321,HP:0100834,Neoplasm of the large intestine,0.206365,0.006946,67
904,HP:0010787,Genital neoplasm,0.133163,0.008309,102
708,HP:0007379,Neoplasm of the genitourinary tract,0.098904,0.009468,139
1093,HP:0100568,Neoplasm of the endocrine system,0.098258,0.009500,86
1178,HP:0002034,Abnormality of the rectum,0.091344,0.005468,74


In [11]:
sets.query('hpoID == "HP:0040181"')

,hpoID,hpolabel,geneID,Gene
297326,HP:0040181,Chapped lip,3853,KRT6A


In [12]:
hpo_t.query('deltaZ < 0 and pValue < 0.01').sort_values('deltaZ').head(10)

,hpoID,hpoLabel,deltaZ,pValue,nGenes
4717,HP:0006549,Unilateral primary pulmonary dysgenesis,-0.175750,0.000000,4
4742,HP:0011590,Double aortic arch,-0.175750,0.000000,4
5626,HP:0006699,Premature atrial contractions,-0.175750,0.000000,3
6223,HP:0040266,Proximal upper limb muscle hypertrophy,-0.175749,0.000000,2
6210,HP:0040265,Upper limb muscle hypertrophy,-0.175749,0.000000,2
6529,HP:0040278,Prolactinoma,-0.175749,0.000000,2
6682,HP:0040025,Clinodactyly of the 4th finger,-0.175749,0.000000,2
6098,HP:0040217,Elevated hemoglobin A1c,-0.175749,0.000000,2
5512,HP:0012398,Peripheral edema,-0.169074,0.000428,3
3965,HP:0001664,Torsade de pointes,-0.158521,0.000093,7


In [13]:
hpo_t.query('deltaZ > 0 and pValue < 0.01 and nGenes < 100').sort_values('deltaZ', ascending=False).head(10)

,hpoID,hpoLabel,deltaZ,pValue,nGenes
6137,HP:0030410,Sebaceous gland carcinoma,1.165369,0.004459,2
6141,HP:0006719,Benign gastrointestinal tract tumors,1.165369,0.004459,2
6280,HP:0006778,Benign genitourinary tract neoplasm,1.165369,0.004459,2
5318,HP:0006758,Malignant genitourinary tract tumor,1.073453,0.007271,3
1612,HP:0100273,Neoplasm of the colon,0.277430,0.007167,49
1321,HP:0100834,Neoplasm of the large intestine,0.206365,0.006946,67
1093,HP:0100568,Neoplasm of the endocrine system,0.098258,0.009500,86
1178,HP:0002034,Abnormality of the rectum,0.091344,0.005468,74
5364,HP:0011939,3-4 finger cutaneous syndactyly,0.076933,0.000902,3
5129,HP:0011784,Thyrotoxicosis with diffuse goiter,0.036587,0.009869,2


In [14]:
sets.query("hpoID == 'HP:0001417' and Gene in @zs.index")

,hpoID,hpolabel,geneID,Gene
496815,HP:0001417,X-linked inheritance,9227,LRAT
496818,HP:0001417,X-linked inheritance,24,ABCA4
496821,HP:0001417,X-linked inheritance,5148,PDE6G
496833,HP:0001417,X-linked inheritance,10804,GJB6
496866,HP:0001417,X-linked inheritance,4724,NDUFS4
496867,HP:0001417,X-linked inheritance,440435,GPR179
496874,HP:0001417,X-linked inheritance,7299,TYR
496876,HP:0001417,X-linked inheritance,84100,ARL6
496885,HP:0001417,X-linked inheritance,2706,GJB2
496907,HP:0001417,X-linked inheritance,4286,MITF
